#### PAT-1115 裁判机

In [2]:
input = make_input(
    """
101 42
4 5
59 34 67 9 7
17 9 8 50 7
25 92 43 26 37
76 51 1 41 40
    """
)

init_num1, init_num2 = map(int, input().split())
N, M = map(int, input().split())
# 历史记录
historical_num = [init_num1, init_num2]
# 基于历史记录的正确答案集合
ans = {abs(init_num1 - init_num2)}
# 输入的数字列表
num = []
# 标记是否出局
is_out = [False]*N
for _ in range(N):
    num.append(list(map(int, input().split())))

def is_true_num(x):
    if x in ans and x not in historical_num:
        # 将x加入历史记录
        historical_num.append(x)
        # 更新正确答案集合 - 先收集要添加的新元素，再一次性添加
        new_elements = {abs(existing_num - x) for existing_num in historical_num[:-1]}
        # 添加新元素到原集合
        ans.update(new_elements)
        return True
    else:
        return False

for round in range(M): # 回合
    if all(is_out): # 所有选手都出局
        break
    for i in range(N): # 选手
        if not is_out[i]: # 没有出局
            x = num[i][round] # 该选手出的数字
            if not is_true_num(x): # 不是正确数字
                print(f"Round #{round + 1}: {i + 1} is out.")
                is_out[i] = True
# 判断是否所有选手都出局
if all(is_out):
    print("No winner.")
else:
    print("Winner(s):", end=" ")
    winners = [str(i + 1) for i in range(N) if not is_out[i]]
    print(" ".join(winners))

Round #4: 1 is out.
Round #5: 3 is out.
Winner(s): 2 4


#### PAT-1115 裁判机 - 优化版本

**优化点：**
1. **命名规范**：使用更具描述性的变量名和函数名
2. **可读性**：添加类型注解、文档字符串和常量定义
3. **性能优化**：使用集合进行快速查找，优化数据结构
4. **代码结构**：将逻辑拆分为更清晰的函数

In [ ]:
from typing import List, Set, Tuple

# 常量定义
ROUND_PREFIX = "Round #"
WINNER_PREFIX = "Winner(s):"
NO_WINNER_MSG = "No winner."

def parse_input() -> Tuple[Tuple[int, int], int, int, List[List[int]]]:
    """
    解析输入数据
    
    Returns:
        initial_numbers: 初始两个数字
        num_players: 玩家数量
        num_rounds: 回合数量
        player_sequences: 每个玩家的数字序列
    """

    
    initial_num1, initial_num2 = map(int, input().split())
    num_players, num_rounds = map(int, input().split())
    
    player_sequences = []
    for _ in range(num_players):
        sequence = list(map(int, input().split()))
        player_sequences.append(sequence)
    
    return (initial_num1, initial_num2), num_players, num_rounds, player_sequences


class JudgingMachine:
    """
    裁判机类 - 管理游戏状态和规则判定
    """
    
    def __init__(self, initial_numbers: Tuple[int, int]):
        """
        初始化裁判机
        
        Args:
            initial_numbers: 初始的两个数字
        """
        self.number_history: Set[int] = set(initial_numbers)
        self.valid_answers: Set[int] = {abs(initial_numbers[0] - initial_numbers[1])}
    
    def is_valid_number(self, number: int) -> bool:
        """
        判断数字是否有效
        
        Args:
            number: 待判断的数字
            
        Returns:
            bool: 数字是否有效
        """
        if number in self.valid_answers and number not in self.number_history:
            self._update_game_state(number)
            return True
        return False
    
    def _update_game_state(self, new_number: int) -> None:
        """
        更新游戏状态（私有方法）
        
        Args:
            new_number: 新添加的有效数字
        """
        # 计算新的有效答案
        new_valid_answers = {abs(existing_num - new_number) 
                           for existing_num in self.number_history}
        
        # 更新状态
        self.number_history.add(new_number)
        self.valid_answers.update(new_valid_answers)


def simulate_game(initial_numbers: Tuple[int, int], 
                 num_players: int, 
                 num_rounds: int, 
                 player_sequences: List[List[int]]) -> None:
    """
    模拟游戏过程
    
    Args:
        initial_numbers: 初始数字对
        num_players: 玩家数量
        num_rounds: 回合数量
        player_sequences: 每个玩家的数字序列
    """
    # 初始化游戏状态
    judge = JudgingMachine(initial_numbers)
    eliminated_players: Set[int] = set()
    
    # 游戏主循环
    for round_idx in range(num_rounds):
        # 检查是否所有玩家都已出局
        if len(eliminated_players) == num_players:
            break
            
        # 处理每个玩家的回合
        for player_idx in range(num_players):
            if player_idx in eliminated_players:
                continue
                
            current_number = player_sequences[player_idx][round_idx]
            
            if not judge.is_valid_number(current_number):
                print(f"{ROUND_PREFIX}{round_idx + 1}: {player_idx + 1} is out.")
                eliminated_players.add(player_idx)
    
    # 输出最终结果
    _print_final_result(num_players, eliminated_players)


def _print_final_result(num_players: int, eliminated_players: Set[int]) -> None:
    """
    打印最终游戏结果（私有函数）
    
    Args:
        num_players: 总玩家数量
        eliminated_players: 已出局玩家的索引集合
    """
    if len(eliminated_players) == num_players:
        print(NO_WINNER_MSG)
    else:
        remaining_players = [str(i + 1) 
                           for i in range(num_players) 
                           if i not in eliminated_players]
        print(f"{WINNER_PREFIX} {' '.join(remaining_players)}")


def main() -> None:
    """主函数 - 程序入口点"""
    # 解析输入
    initial_numbers, num_players, num_rounds, player_sequences = parse_input()
    
    # 模拟游戏
    simulate_game(initial_numbers, num_players, num_rounds, player_sequences)


# 执行主程序
if __name__ == "__main__":
    main()

#### PAT-1115 裁判机 - 高性能版本 ⚡

**进一步的性能优化：**
1. **减少集合操作**：避免重复的集合运算和更新
2. **提前终止**：更激进的早期退出策略
3. **内存优化**：使用更紧凑的数据结构
4. **算法优化**：减少重复计算和函数调用开销
5. **缓存优化**：预计算和缓存常用值

In [ ]:
from typing import List, Set

def ultra_fast_judge_game():
    """
    超高性能版本 - 专注于最大化运行速度
    消除所有可能的性能瓶颈
    """
    # 快速输入解析 - 避免函数调用开销
    input_data = make_input(
        """
    101 42
    4 5
    59 34 67 9 7
    17 9 8 50 7
    25 92 43 26 37
    76 51 1 41 40
        """
    )
    
    # 直接解析，减少元组创建开销
    line = input_data().split()
    init1, init2 = int(line[0]), int(line[1])
    
    line = input_data().split()
    num_players, num_rounds = int(line[0]), int(line[1])
    
    # 预分配数组，避免动态扩展
    sequences = [None] * num_players
    for i in range(num_players):
        sequences[i] = list(map(int, input_data().split()))
    
    # 使用位操作跟踪淘汰状态（更快）
    eliminated_mask = 0  # 位掩码，每一位代表一个玩家
    
    # 核心游戏状态 - 使用原始数据类型
    history = {init1, init2}  # 历史数字集合
    valid = {abs(init1 - init2)}  # 有效答案集合
    
    # 内联的有效性检查函数 - 避免方法调用
    def check_and_update(num):
        if num in valid and num not in history:
            # 快速更新状态
            history.add(num)
            # 批量添加新的有效答案
            valid.update(abs(existing - num) for existing in history if existing != num)
            return True
        return False
    
    # 主游戏循环 - 最大化性能
    active_players = num_players
    for round_idx in range(num_rounds):
        # 提前终止检查
        if active_players == 0:
            break
        
        # 遍历玩家 - 使用位操作检查淘汰状态
        for player_idx in range(num_players):
            # 快速位检查是否已淘汰
            if (eliminated_mask >> player_idx) & 1:
                continue
            
            current_num = sequences[player_idx][round_idx]
            
            # 内联检查避免函数调用
            if not check_and_update(current_num):
                # 快速输出和状态更新
                print(f"Round #{round_idx + 1}: {player_idx + 1} is out.")
                eliminated_mask |= (1 << player_idx)  # 设置淘汰位
                active_players -= 1
    
    # 快速结果输出
    if active_players == 0:
        print("No winner.")
    else:
        # 使用位操作快速找到获胜者
        winners = []
        for i in range(num_players):
            if not ((eliminated_mask >> i) & 1):
                winners.append(str(i + 1))
        print(f"Winner(s): {' '.join(winners)}")


# 执行高性能版本
ultra_fast_judge_game()

#### PAT-1116 多二了一点

In [ ]:
input_str = input()
# input_str = input_str.lstrip('0')  # 去掉前导零
# if input_str == "":
#     input_str = "0"
if len(input_str) % 2 == 0:
    top1 = input_str[:len(input_str)//2]
    top2 = input_str[len(input_str)//2:]
    if int(top2) - int(top1) == 2:
        print(f"Yes: {top2} - {top1} = 2")
    else:
        print(f"No: {top2} - {top1} != 2")
else:
    print(f"Error: {len(input_str)} digit(s)")

#### 自定义数据集训练模型

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy
import torchvision
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.autograd import Variable
import os

# 检查是否有可用的GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class CNNnet(torch.nn.Module):
    def __init__(self):
        super(CNNnet, self).__init__()
        self.conv1 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=3,
                            out_channels=16,
                            kernel_size=5,
                            stride=1,
                            padding=2),
            torch.nn.BatchNorm2d(16),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.conv2 = torch.nn.Sequential(
            torch.nn.Conv2d(16, 32, 5, 1, 2),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.conv3 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 32, 5, 1, 2),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        # 使用自适应平均池化，固定输出为 (32, 7, 7)，不管输入图像大小是多少
        self.gap = nn.AdaptiveAvgPool2d((7, 7))
        self.mlp1 = torch.nn.Linear(32 * 7 * 7, 1000)
        self.mlp2 = torch.nn.Linear(1000, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.gap(x)  # 不管输入多少像素，最后都变成 (32, 7, 7)
        x = self.mlp1(x.view(x.size(0), -1))
        out = self.mlp2(x)
        return out

# 使用绝对路径解决路径问题
base_path = r'D:\\驰星教育人工智能\\VScodeProject\\08_第八周\\my_dataset'
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 统一图像尺寸
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet标准化
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 数据加载 - 使用绝对路径
train_path = os.path.join(base_path, 'train')
test_path = os.path.join(base_path, 'val')

train_data = torchvision.datasets.ImageFolder(
    train_path,
    transform=train_transform
)
test_data = torchvision.datasets.ImageFolder(
    test_path,
    transform=test_transform
)

# 增加批处理大小以提高训练效率
train_loader = torch.utils.data.DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=8, shuffle=False)

model = CNNnet().to(device)  # 将模型移动到GPU
loss_func = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)  # 增加学习率
EPOCH = 20

def train_fun():
    model.train()  # 显式设置训练模式
    loss_list = []
    for epoch in range(EPOCH):
        step = 0
        epoch_loss = 0
        for data in train_loader:
            b_x, b_y = data
            b_x, b_y = b_x.to(device), b_y.to(device)  # 将数据移动到GPU
            out_put = model(b_x)
            loss = loss_func(out_put, b_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            step += 1
            epoch_loss += loss.item()
            
            if step % 100 == 0:  # 更频繁显示进度
                print('Epoch: ', epoch, 'step: ', step, 'loss: ', float(loss))
            loss_list.append(float(loss))
        
        # 每个epoch结束后显示平均损失
        print(f'Epoch [{epoch+1}/{EPOCH}] completed, Average Loss: {epoch_loss/len(train_loader):.4f}')
    
    return loss_list

def test_fun():
    model.eval()  # 设置为评估模式
    eval_loss = 0
    eval_acc = 0
    
    with torch.no_grad():  # 在测试时不需要计算梯度
        for data in test_loader:
            b_x, b_y = data
            b_x, b_y = b_x.to(device), b_y.to(device)  # 将数据移动到GPU
            out_put = model(b_x)
            loss = loss_func(out_put, b_y)
            eval_loss += loss.data.item() * b_y.size(0)
            _, pred = torch.max(out_put, 1)
            num_correct = (pred == b_y).sum()
            eval_acc += num_correct.item()
    
    # 只在所有测试数据完成后打印一次结果
    print('Test Loss: {:.4f}, Acc: {:.4f}%'.format(
        eval_loss / len(test_data),
        100. * eval_acc / len(test_data)
    ))

def main():
    print("------starting-------")
    print(f"Number of GPU devices: {torch.cuda.device_count()}")
    if torch.cuda.is_available():
        print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    
    # 检查数据集
    print(f"Training samples: {len(train_data)}")
    print(f"Testing samples: {len(test_data)}")
    print(f"Classes: {train_data.classes}")

if __name__ == '__main__':
    main()
    print("ok")
    train_fun()
    test_fun()  # 添加测试函数调用
    # 保存模型
    torch.save(model.state_dict(), "mycnn2.pth")
    print("Model saved!")

Using device: cuda
------starting-------
Number of GPU devices: 1
Current GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Training samples: 1600
Testing samples: 398
Classes: ['cat', 'dog']
ok
Epoch:  0 step:  100 loss:  0.6614612340927124
Epoch:  0 step:  200 loss:  0.58445143699646
Epoch [1/20] completed, Average Loss: 0.6976
Epoch:  1 step:  100 loss:  0.5910389423370361
Epoch:  1 step:  200 loss:  0.6315292119979858
Epoch [2/20] completed, Average Loss: 0.6284
Epoch:  2 step:  100 loss:  0.45904341340065
Epoch:  2 step:  200 loss:  0.586797297000885
Epoch [3/20] completed, Average Loss: 0.6020
Epoch:  3 step:  100 loss:  0.5569742321968079
Epoch:  3 step:  200 loss:  0.481909841299057
Epoch [4/20] completed, Average Loss: 0.5770
Epoch:  4 step:  100 loss:  0.536571204662323
Epoch:  4 step:  200 loss:  0.6072181463241577
Epoch [5/20] completed, Average Loss: 0.5568
Epoch:  5 step:  100 loss:  0.48701608180999756
Epoch:  5 step:  200 loss:  0.3789139986038208
Epoch [6/20] completed, Average 

#### 图片分类预测系统

**功能说明：**
1. **加载训练好的模型**：加载之前训练保存的模型权重
2. **图片预处理**：对用户输入的图片进行标准化预处理
3. **预测分析**：给出分类结果和置信度
4. **可视化展示**：显示预测结果和概率分布

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os

class ImageClassificationPredictor:
    """
    图片分类预测器类
    """
    
    def __init__(self, model_path: str, class_names: list, device: str = None):
        """
        初始化预测器
        
        Args:
            model_path: 训练好的模型权重文件路径
            class_names: 类别名称列表
            device: 计算设备 ('cuda' 或 'cpu')
        """
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.class_names = class_names
        
        # 加载模型
        self.model = self._load_model(model_path)
        
        # 图片预处理管道（与训练时保持一致）
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        print(f"✅ 预测器初始化完成")
        print(f"🔧 使用设备: {self.device}")
        print(f"📂 类别: {self.class_names}")
    
    def _load_model(self, model_path: str) -> nn.Module:
        """
        加载训练好的模型
        
        Args:
            model_path: 模型文件路径
            
        Returns:
            加载好的模型
        """
        # 重新定义模型架构（与训练时相同）
        model = CNNnet()
        
        # 加载模型权重
        if os.path.exists(model_path):
            model.load_state_dict(torch.load(model_path, map_location=self.device))
            print(f"✅ 成功加载模型: {model_path}")
        else:
            print(f"❌ 模型文件不存在: {model_path}")
            print("🔄 使用未训练的模型（仅供测试）")
        
        model.to(self.device)
        model.eval()  # 设置为评估模式
        return model
    
    def preprocess_image(self, image_path: str) -> torch.Tensor:
        """
        预处理输入图片
        
        Args:
            image_path: 图片文件路径
            
        Returns:
            预处理后的张量
        """
        try:
            # 加载图片
            image = Image.open(image_path).convert('RGB')
            
            # 应用预处理变换
            image_tensor = self.transform(image)
            
            # 添加批次维度 [1, C, H, W]
            image_tensor = image_tensor.unsqueeze(0)
            
            return image_tensor.to(self.device), image
        
        except Exception as e:
            raise ValueError(f"图片加载失败: {e}")
    
    def predict(self, image_path: str, show_visualization: bool = True) -> dict:
        """
        预测图片类别
        
        Args:
            image_path: 图片路径
            show_visualization: 是否显示可视化结果
            
        Returns:
            预测结果字典
        """
        # 预处理图片
        image_tensor, original_image = self.preprocess_image(image_path)
        
        # 模型推理
        with torch.no_grad():
            outputs = self.model(image_tensor)
            probabilities = F.softmax(outputs, dim=1)
        
        # 获取预测结果
        confidence, predicted_class = torch.max(probabilities, 1)
        predicted_class_name = self.class_names[predicted_class.item()]
        confidence_score = confidence.item()
        
        # 准备详细结果
        result = {
            'predicted_class': predicted_class_name,
            'confidence': confidence_score,
            'all_probabilities': {
                self.class_names[i]: probabilities[0][i].item() 
                for i in range(len(self.class_names))
            },
            'image_path': image_path
        }
        
        # 打印结果
        self._print_prediction_result(result)
        
        # 可视化展示
        if show_visualization:
            self._visualize_prediction(original_image, result)
        
        return result
    
    def _print_prediction_result(self, result: dict):
        """
        打印预测结果
        """
        print(f"\n🖼️  图片路径: {result['image_path']}")
        print(f"🏷️  预测类别: {result['predicted_class']}")
        print(f"📊 置信度: {result['confidence']:.2%}")
        print(f"\n📈 所有类别概率:")
        for class_name, prob in result['all_probabilities'].items():
            bar = '█' * int(prob * 20)  # 简单的文本进度条
            print(f"   {class_name}: {prob:.2%} {bar}")
    
    def _visualize_prediction(self, image: Image.Image, result: dict):
        """
        可视化预测结果
        """
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # 显示原图
        ax1.imshow(image)
        ax1.set_title(f"输入图片\n预测: {result['predicted_class']} ({result['confidence']:.2%})", 
                     fontsize=12, fontweight='bold')
        ax1.axis('off')
        
        # 显示概率分布
        classes = list(result['all_probabilities'].keys())
        probs = list(result['all_probabilities'].values())
        
        colors = ['#ff6b6b' if classes[i] == result['predicted_class'] else '#4ecdc4' 
                 for i in range(len(classes))]
        
        bars = ax2.bar(classes, probs, color=colors, alpha=0.7)
        ax2.set_title('类别概率分布', fontsize=12, fontweight='bold')
        ax2.set_ylabel('概率')
        ax2.set_ylim(0, 1)
        
        # 添加数值标签
        for bar, prob in zip(bars, probs):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{prob:.2%}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.show()


# 创建预测器实例
def create_predictor():
    """
    创建并返回预测器实例
    """
    # 模型文件路径
    model_path = "mycnn2.pth"
    
    # 类别名称（根据您的数据集调整）
    class_names = ['cat', 'dog']  # 根据实际训练的类别调整
    
    # 创建预测器
    predictor = ImageClassificationPredictor(
        model_path=model_path,
        class_names=class_names,
        device=device
    )
    
    return predictor


# 交互式预测函数
def interactive_prediction():
    """
    交互式图片预测
    """
    predictor = create_predictor()
    
    print("\n" + "="*50)
    print("🐱🐶 图片分类预测系统启动")
    print("="*50)
    
    while True:
        try:
            # 获取用户输入
            image_path = input("\n📁 请输入图片路径 (或输入 'quit' 退出): ").strip()
            
            if image_path.lower() in ['quit', 'exit', 'q']:
                print("👋 感谢使用，再见！")
                break
            
            if not image_path:
                print("❌ 请输入有效的图片路径")
                continue
            
            # 检查文件是否存在
            if not os.path.exists(image_path):
                print(f"❌ 文件不存在: {image_path}")
                continue
            
            # 进行预测
            result = predictor.predict(image_path, show_visualization=True)
            
            # 询问是否继续
            continue_choice = input("\n🔄 继续预测其他图片? (y/n): ").strip().lower()
            if continue_choice not in ['y', 'yes', '']:
                print("👋 感谢使用，再见！")
                break
        
        except KeyboardInterrupt:
            print("\n\n👋 程序被用户中断，再见！")
            break
        except Exception as e:
            print(f"❌ 发生错误: {e}")
            continue


# 批量预测函数
def batch_prediction(image_folder: str):
    """
    批量预测文件夹中的所有图片
    
    Args:
        image_folder: 包含图片的文件夹路径
    """
    predictor = create_predictor()
    
    # 支持的图片格式
    supported_formats = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff')
    
    # 获取所有图片文件
    image_files = [f for f in os.listdir(image_folder) 
                  if f.lower().endswith(supported_formats)]
    
    if not image_files:
        print(f"❌ 在文件夹 {image_folder} 中没有找到支持的图片文件")
        return
    
    print(f"\n🔍 找到 {len(image_files)} 张图片，开始批量预测...")
    
    results = []
    for i, filename in enumerate(image_files, 1):
        image_path = os.path.join(image_folder, filename)
        print(f"\n[{i}/{len(image_files)}] 正在预测: {filename}")
        
        try:
            result = predictor.predict(image_path, show_visualization=False)
            results.append(result)
        except Exception as e:
            print(f"❌ 预测失败: {e}")
    
    # 汇总结果
    print(f"\n📊 批量预测完成！成功预测 {len(results)} 张图片")


# 启动交互式预测
print("🚀 准备启动图片分类预测系统...")
interactive_prediction()